# Local evaluator

Fast end-to-end smoke test for the Biohub UNet3D + transformer + ILP pipeline. It selects the shortest labeled training video, predicts its graph, and evaluates it with the metric implementation shipped in the public support artifact.

**Interpretation warning:** this run proves that inference and scoring work. It is not yet an unbiased validation score because the public checkpoint may have seen this video, and this first version evaluates the raw predicted graph before the leaderboard notebook's additional graph repair.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

COMPETITION = 'biohub-cell-tracking-during-development'
TRAIN_DIR = Path(f'/kaggle/input/competitions/{COMPETITION}/train')
WORK_DIR = Path('/kaggle/working/biohub_local_eval')
REPO_DIR = WORK_DIR / 'repo'
DET_THRESHOLD = 0.97
UNET_BATCH_SIZE = 4
MAX_DISTANCE_UM = 7.0

assert TRAIN_DIR.exists(), f'Missing competition train mount: {TRAIN_DIR}'
print('TRAIN_DIR:', TRAIN_DIR)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
# Locate the attached pilkwang support artifact.
manifest_candidates = list(Path('/kaggle/input').glob('**/ARTIFACT_MANIFEST.json'))
support_manifest = None
for candidate in manifest_candidates:
    try:
        meta = json.loads(candidate.read_text())
    except Exception:
        continue
    if meta.get('model', {}).get('method') == 'unet_transformer' and (candidate.parent / 'repo').exists():
        support_manifest = candidate
        break
assert support_manifest is not None, 'Attach pilkwang/biohub-tracking-support-pack-50ep-v1'
SUPPORT_DIR = support_manifest.parent
manifest = json.loads(support_manifest.read_text())
print('Support:', SUPPORT_DIR)
print('Artifact:', manifest.get('artifact_name'))
print('Weight sha256:', manifest.get('model', {}).get('weight_sha256'))

shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)
if (SUPPORT_DIR / 'repo').is_dir():
    shutil.copytree(SUPPORT_DIR / 'repo', REPO_DIR)
else:
    REPO_DIR.mkdir()
    with zipfile.ZipFile(SUPPORT_DIR / 'repo.zip') as archive:
        archive.extractall(REPO_DIR)

weights_src = SUPPORT_DIR / 'weights'
if weights_src.is_dir():
    shutil.copytree(weights_src, REPO_DIR / 'weights')
else:
    (REPO_DIR / 'weights').mkdir()
    with zipfile.ZipFile(SUPPORT_DIR / 'weights.zip') as archive:
        archive.extractall(REPO_DIR / 'weights')
print('Materialized repo:', REPO_DIR)

In [ ]:
# Build an isolated, internally consistent runtime without venv/ensurepip.
# Kaggle's /usr/bin/python3 cannot create venvs, so wheels go into a private --target directory.
RUNTIME_SITE = WORK_DIR / 'runtime_site'
RUNTIME_SITE.mkdir(parents=True, exist_ok=True)
RUNTIME_PYTHON = sys.executable
wheels_dir = SUPPORT_DIR / 'wheels'
assert wheels_dir.exists(), f'Missing offline wheels: {wheels_dir}'
wheel_files = sorted(wheels_dir.glob('*.whl'))
assert wheel_files, f'No wheels found in {wheels_dir}'

install_cmd = [
    RUNTIME_PYTHON, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
    '--upgrade', '--force-reinstall', '--target', str(RUNTIME_SITE),
    *map(str, wheel_files),
]
installed = subprocess.run(install_cmd, text=True, capture_output=True)
assert installed.returncode == 0, 'Private runtime install failed:\n' + installed.stderr[-12000:]

runtime_env = {
    **os.environ,
    'PYTHONPATH': os.pathsep.join([str(RUNTIME_SITE), str(REPO_DIR / 'src')]),
    'PYTHONNOUSERSITE': '1',
}
probe = subprocess.run(
    [RUNTIME_PYTHON, '-c',
     'import numpy, scipy, scipy.spatial, numcodecs, zarr, geff_spec, tracksdata; '
     'print(numpy.__version__, scipy.__version__, numcodecs.__version__)'],
    env=runtime_env, text=True, capture_output=True,
)
assert probe.returncode == 0, 'Private runtime check failed:\n' + probe.stderr[-12000:]
print('Private runtime ready; NumPy/SciPy/numcodecs:', probe.stdout.strip())


In [ ]:
# Pick the shortest labeled video to minimize smoke-test runtime.
def video_shape(path):
    return tuple(json.loads((path / '0' / 'zarr.json').read_text())['shape'])

candidates = []
for zarr_path in sorted(TRAIN_DIR.glob('*.zarr')):
    if (TRAIN_DIR / f'{zarr_path.stem}.geff').exists():
        shape = video_shape(zarr_path)
        candidates.append((shape[0], int(shape[1] * shape[2] * shape[3]), zarr_path.stem, shape))
assert candidates, 'No paired .zarr/.geff training datasets found'
_, _, SAMPLE, SAMPLE_SHAPE = min(candidates)
print(f'Selected {SAMPLE}: shape={SAMPLE_SHAPE}; labeled candidates={len(candidates)}')

splits_path = REPO_DIR / 'local_smoke_split.json'
splits_path.write_text(json.dumps([{'split': 0, 'train': [], 'test': [SAMPLE]}], indent=2))
weights = REPO_DIR / 'weights/unet_transformer/split_0/edge_predictor_best.pth'
assert weights.exists(), weights

In [ ]:
# One-video UNet + transformer + ILP inference in the isolated runtime.
env = {
    **runtime_env,
    'USER': 'local_eval',
    'USERNAME': 'local_eval',
    'BIOHUB_DATA_DIR': str(TRAIN_DIR),
}
cmd = [
    RUNTIME_PYTHON, 'scripts/predict_unet_transformer.py',
    '--data-dir', str(TRAIN_DIR),
    '--splits', str(splits_path),
    '--split', '0',
    '--weights', str(weights),
    '--unet-batch-size', str(UNET_BATCH_SIZE),
    '--det-threshold', str(DET_THRESHOLD),
    '--use-ilp',
]
print(' '.join(cmd))
started = time.time()
run = subprocess.run(cmd, cwd=REPO_DIR, env=env, text=True, capture_output=True)
print(run.stdout)
assert run.returncode == 0, 'Inference failed:\n' + run.stderr[-12000:]
runtime_minutes = (time.time() - started) / 60
print(f'Inference finished in {runtime_minutes:.2f} min')


In [ ]:
# Evaluate in the same isolated runtime. The main notebook kernel imports no scientific stack.
eval_cmd = [
    RUNTIME_PYTHON, 'scripts/evaluate.py',
    '--method', 'unet_transformer',
    '--split', '0',
    '--max-distance', str(MAX_DISTANCE_UM),
]
evaluated = subprocess.run(
    eval_cmd, cwd=REPO_DIR, env=env, text=True, capture_output=True,
)
print(evaluated.stdout)
assert evaluated.returncode == 0, 'Evaluation failed:\n' + evaluated.stderr[-12000:]

import re
match = re.search(
    r'score=([0-9.]+).*edge_jaccard=([0-9.]+).*adj_edge_jaccard=([0-9.]+).*division_jaccard=([0-9.]+).*node_recall=([0-9.]+)',
    evaluated.stdout,
)
assert match, 'Evaluator completed but its summary could not be parsed:\n' + evaluated.stdout[-4000:]
keys = ['score', 'edge_jaccard', 'adjusted_edge_jaccard', 'division_jaccard', 'node_recall']
metrics = {key: float(value) for key, value in zip(keys, match.groups())}
result_payload = {
    'dataset': SAMPLE,
    'shape': SAMPLE_SHAPE,
    'runtime_minutes': runtime_minutes,
    'weight_sha256': manifest.get('model', {}).get('weight_sha256'),
    **metrics,
}
print(json.dumps(result_payload, indent=2))
result_path = Path('/kaggle/working/local_evaluator_result.json')
result_path.write_text(json.dumps(result_payload, indent=2))
print('Saved:', result_path)


## Next validation step

Once this smoke test succeeds, the next version will use embryo-grouped holdouts and fold-specific checkpoints. Never use this single training-video score to select hyperparameters.